# PDF Ingestion & Indexing Pipeline

Loads department PDFs, splits them into **heading/section-based** chunks (not fixed page
sizes), embeds them with OpenAI, and upserts them into **Pinecone** with rich metadata:
`department`, `document_name`, `section_title`, page range, and chunk index.

**Expected directory structure:**
```
documents/
  departments/
    hr/
      policy_handbook.pdf
      leave_policy.pdf
    finance/
      expense_procedure.pdf
    it/
      access_management.pdf
```

Re-running this notebook is safe: a local manifest (`indexed_manifest.json`) tracks file
hashes so unchanged PDFs are skipped, and chunk IDs are deterministic so re-indexing a
changed file overwrites its old vectors instead of duplicating them.

**Requirements:** `pip install pymupdf tiktoken pinecone-client openai python-dotenv tqdm`

**.env file needed (same folder):**
```
OPENAI_API_KEY=sk-...
PINECONE_API_KEY=pcsk-...
```


In [1]:
import os
import re
import json
import hashlib
from pathlib import Path

import fitz  # PyMuPDF
import tiktoken
from tqdm.auto import tqdm
from dotenv import load_dotenv

from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

# ---------------- Configuration ----------------
DOCUMENTS_ROOT = Path("documents/internal_docs_by_area")

PINECONE_INDEX_NAME = "ironstore-enterprise-knowledge-base"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIM = 3072

MAX_CHUNK_TOKENS = 500        # target tokens per chunk (only used if a section is long)
CHUNK_OVERLAP_TOKENS = 75     # overlap between sub-chunks of a long section

MANIFEST_PATH = Path("indexed_manifest.json")  # tracks file hashes for incremental re-indexing

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
tokenizer = tiktoken.get_encoding("cl100k_base")


c:\Users\kriti\Dropbox\PC\Desktop\Ironhack_AI_Engineering\langchain-v0.2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Section detection

Instead of chunking by fixed page/character sizes, we use **font-size and boldness
heuristics** (via PyMuPDF) plus common heading patterns ("1. Introduction", "Section 2:",
ALL CAPS lines) to detect section headings, then group the text under each heading into
one logical section.


In [2]:
def _file_hash(path: Path) -> str:
    h = hashlib.md5()
    h.update(path.read_bytes())
    return h.hexdigest()


def extract_sections_from_pdf(pdf_path: Path):
    """
    Extract text from a PDF and group it into sections based on detected headings
    (font size / boldness / numbering heuristics), rather than raw pages.

    Returns a list of dicts: {"heading", "text", "start_page", "end_page"}
    """
    doc = fitz.open(pdf_path)

    # First pass: find the most common font size -> treat it as "body text" baseline
    all_sizes = []
    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    if span["text"].strip():
                        all_sizes.append(round(span["size"], 1))

    if not all_sizes:
        doc.close()
        return []

    body_size = max(set(all_sizes), key=all_sizes.count)
    heading_size_threshold = body_size + 1.5

    heading_pattern = re.compile(
        r"^(chapter|section|appendix)?\s*\d+(\.\d+)*[\.\)]?\s+.+", re.IGNORECASE
    )

    def is_heading(span_text, size, flags):
        is_bold = bool(flags & 2 ** 4)          # bold flag bit in PyMuPDF span flags
        big_enough = size >= heading_size_threshold
        short_line = len(span_text.split()) <= 12
        numbered = bool(heading_pattern.match(span_text.strip()))
        all_caps = span_text.strip().isupper() and len(span_text.strip()) > 3
        return short_line and (big_enough or (is_bold and big_enough) or numbered or all_caps)

    sections = []
    current_heading = "Introduction"
    current_text_parts = []
    current_start_page = 1

    for page_num, page in enumerate(doc, start=1):
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                line_text = "".join(span["text"] for span in line.get("spans", [])).strip()
                if not line_text:
                    continue
                span = line["spans"][0]

                if is_heading(line_text, span["size"], span["flags"]):
                    if current_text_parts:
                        sections.append({
                            "heading": current_heading,
                            "text": "\n".join(current_text_parts).strip(),
                            "start_page": current_start_page,
                            "end_page": page_num,
                        })
                    current_heading = line_text
                    current_text_parts = []
                    current_start_page = page_num
                else:
                    current_text_parts.append(line_text)

    if current_text_parts:
        sections.append({
            "heading": current_heading,
            "text": "\n".join(current_text_parts).strip(),
            "start_page": current_start_page,
            "end_page": len(doc),
        })

    doc.close()
    # drop near-empty sections (e.g. stray headers with no real content)
    return [s for s in sections if len(s["text"].split()) >= 10]


## 2. Sub-chunking long sections

Most sections (procedures, policy clauses) are kept as a **single chunk** so the heading
and its full content stay together for the LLM. Only sections that exceed
`MAX_CHUNK_TOKENS` get split further, with a token overlap so context isn't lost at the
split boundary.


In [3]:
def chunk_section_text(text: str, max_tokens=MAX_CHUNK_TOKENS, overlap=CHUNK_OVERLAP_TOKENS):
    """Split a section's text into token-bounded chunks with overlap, only if needed."""
    tokens = tokenizer.encode(text)
    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end]))
        if end == len(tokens):
            break
        start = end - overlap
    return chunks


## 3. Build chunk records with metadata

Walks `documents/departments/<department>/<file>.pdf`, extracts sections, sub-chunks if
needed, and attaches metadata (`department`, `document_name`, `section_title`, page range)
to every chunk. Skips files unchanged since the last run (tracked via `indexed_manifest.json`).


In [4]:
def build_chunk_records(documents_root: Path = DOCUMENTS_ROOT):
    records = []
    manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
    new_manifest = dict(manifest)

    pdf_paths = sorted(documents_root.glob("*/*.pdf"))
    for pdf_path in tqdm(pdf_paths, desc="Scanning PDFs"):
        department = pdf_path.parent.name
        document_name = pdf_path.name
        file_key = str(pdf_path)
        current_hash = _file_hash(pdf_path)

        if manifest.get(file_key) == current_hash:
            continue  # unchanged since last run

        sections = extract_sections_from_pdf(pdf_path)
        for sec_idx, section in enumerate(sections):
            sub_chunks = chunk_section_text(section["text"])
            for chunk_idx, chunk_text in enumerate(sub_chunks):
                chunk_id_raw = f"{department}|{document_name}|{sec_idx}|{chunk_idx}"
                chunk_id = hashlib.md5(chunk_id_raw.encode()).hexdigest()
                records.append({
                    "id": chunk_id,
                    "text": chunk_text,
                    "metadata": {
                        "department": department,
                        "document_name": document_name,
                        "section_title": section["heading"],
                        "section_index": sec_idx,
                        "chunk_index": chunk_idx,
                        "start_page": section["start_page"],
                        "end_page": section["end_page"],
                        "source_path": file_key,
                    },
                })
        new_manifest[file_key] = current_hash

    return records, new_manifest


## 4. Pinecone index setup

In [5]:
def get_or_create_index():
    existing = [idx["name"] for idx in pc.list_indexes()]
    if PINECONE_INDEX_NAME not in existing:
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=EMBEDDING_DIM,
            metric="cosine",
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
    return pc.Index(PINECONE_INDEX_NAME)

index = get_or_create_index()


## 5. Embed & upsert

Chunks are embedded in batches and upserted into **per-department namespaces** in
Pinecone. This lets the retrieval step filter by department cheaply, and would make it
straightforward to later restrict a user to only the departments they have access to.


In [10]:
def embed_texts(texts, batch_size=100):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i + batch_size]
        response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        embeddings.extend([item.embedding for item in response.data])
    return embeddings


def upsert_records(records, batch_size=100):
    by_department = {}
    for r in records:
        by_department.setdefault(r["metadata"]["department"], []).append(r)

    for department, dept_records in by_department.items():
        texts = [r["text"] for r in dept_records]
        vectors = embed_texts(texts)

        to_upsert = []
        for r, vec in zip(dept_records, vectors):
            meta = dict(r["metadata"])
            meta["text"] = r["text"]  # keep the raw text in metadata for display at retrieval time
            to_upsert.append((r["id"], vec, meta))

        for i in tqdm(range(0, len(to_upsert), batch_size), desc=f"Upserting [{department}]"):
            index.upsert(vectors=to_upsert[i:i + batch_size], namespace=department)


## 6. Run indexing

In [11]:
records, new_manifest = build_chunk_records()

changed_files = len(set(r["metadata"]["source_path"] for r in records))
print(f"Built {len(records)} chunk records from {changed_files} new/changed PDF(s).")

if records:
    upsert_records(records)
    MANIFEST_PATH.write_text(json.dumps(new_manifest, indent=2))
    print("Indexing complete. Manifest updated.")
else:
    print("No new or changed documents found. Nothing to index.")


Scanning PDFs: 100%|██████████| 20/20 [00:01<00:00, 13.50it/s]


Built 347 chunk records from 20 new/changed PDF(s).


Upserting [Sales]: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

Indexing complete. Manifest updated.


In [13]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=3072, total_vector_count=347, metric='cosine', namespaces=7)

## 7. Verify

In [12]:
stats = index.describe_index_stats()
print(json.dumps(stats.to_dict(), indent=2))


{
  "namespaces": {
    "Customer_support": {
      "vector_count": 54
    },
    "Deliveries": {
      "vector_count": 53
    },
    "Sales": {
      "vector_count": 46
    },
    "IT": {
      "vector_count": 32
    },
    "Finance": {
      "vector_count": 50
    },
    "Legal_Terms": {
      "vector_count": 50
    },
    "HR_policies": {
      "vector_count": 62
    }
  },
  "dimension": 3072,
  "index_fullness": 0.0,
  "total_vector_count": 347,
  "metric": "cosine",
  "vector_type": "dense",
  "memory_fullness": 0.0,
  "storage_fullness": 0.0,
  "response_info": {
    "raw_headers": {
      "date": "Sun, 02 Aug 2026 22:20:28 GMT",
      "content-type": "application/json",
      "content-length": "368",
      "connection": "keep-alive",
      "x-pinecone-request-latency-ms": "40",
      "x-envoy-upstream-service-time": "40",
      "x-pinecone-response-duration-ms": "42",
      "grpc-status": "0",
      "server": "envoy"
    }
  }
}
